<div align="center">

# Day 03: Explaining Deep Networks

**Explainable Artificial Intelligence (VTR UGE 21), hands-on notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Day overview](https://github.com/utkukose/xai-veltech-NB-lecture/tree/main/days/day-03) · [Lecture page](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-03/lecture.html) · [Interactive lab](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-03/lab.html) · [PDF notes](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-03/Day03_Lecture_Notes.pdf)

</div>

## How to use this notebook

This notebook is the hands-on part of the day. The concepts are explained on the [lecture page](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-03/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows; they have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from a Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures, in a few seconds to a few minutes. When the notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Colab installs the three explanation libraries; everything else is preinstalled.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'shap', 'lime', 'gemex'], check=False)

import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (7, 3.6), 'axes.grid': True, 'grid.alpha': 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback. Without an expected value in the
    call, check takes the value and its tolerance from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None with your answer and run the cell again.")
        return False
    try:
        ok = abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected)))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else "not yet. Compare your steps with the lecture and try again."))
    return ok


print('ready')

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 3.1": (lambda: 0.25, 1e-6),
    "Exercise 3.2": (lambda: -1.2, 1e-6),
    "Exercise 3.3": (lambda: 0.0, 1e-6),
    "Exercise A": (lambda: abs(float(ig.sum()) - (fx - fb)), 1e-6),
    "Exercise B": (lambda: score.marker_density_ratio.idxmax(), 1e-6),
}

## Python step 3: A neural network as arrays and the chain rule

A neural network is a sequence of matrix products and simple functions. This step computes a neuron and a two-layer network with NumPy, takes their gradients by the chain rule, checks one of them with finite differences and ends with a layer written as a class &#91;[7](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-03/lecture.html#ref-7)&#93;.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/xai-veltech-NB-lecture/main/days/day-03/NB03_explaining_deep_networks.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

### One neuron

In [ ]:
# Python practice: a small example for learning the language
import numpy as np
x = np.array([0.5, -1.0, 2.0])         # one input with three features
w = np.array([0.8, 0.2, -0.5]); b = 0.1
z = x @ w + b                          # the weighted sum
p = 1 / (1 + np.exp(-z))               # the sigmoid
print("z =", round(float(z), 3), "  p =", round(float(p), 4))

`@` is the matrix product; between two vectors it is the dot product. The sigmoid squeezes the weighted sum into a probability between 0 and 1.

### The gradient by the chain rule

In [ ]:
# Python practice: a small example for learning the language
grad_x = p * (1 - p) * w                # dp/dx = sigmoid'(z) * dz/dx
eps = 1e-6
numeric = np.array([(1 / (1 + np.exp(-((x + eps * np.eye(3)[i]) @ w + b))) - p) / eps for i in range(3)])
print("chain rule        :", grad_x.round(5))
print("finite difference :", numeric.round(5))

The derivative of the sigmoid is $p(1-p)$, and the derivative of the weighted sum with respect to $x$ is $w$; the chain rule multiplies them. Moving one feature by a tiny amount and measuring the change of the output gives the same numbers, which is how every backward pass of the day is checked.

### A two-layer ReLU network

In [ ]:
# Python practice: a small example for learning the language
W1 = np.array([[1.0, -1.0], [0.5, 0.5], [-0.3, 0.8]])   # 3 inputs -> 2 hidden units
w2 = np.array([1.0, -2.0])                              # 2 hidden units -> 1 output
h = np.maximum(0, x @ W1)                               # ReLU, no biases
out = h @ w2
print("hidden:", h, "  output:", round(float(out), 3))

`np.maximum(0, ...)` is the ReLU, the rectified linear unit: It keeps positive values and sets negative ones to zero. The first hidden unit is inactive for this input.

### Its gradient, and gradient times input

In [ ]:
# Python practice: a small example for learning the language
grad = W1 @ (w2 * (h > 0))           # chain rule through the ReLU: inactive units pass nothing back
print("gradient           :", grad)
print("gradient times input:", (grad * x).round(3), "  sum:", round(float((grad * x).sum()), 3))

The ReLU passes the gradient only through active units, which `(h > 0)` expresses as a mask. For a ReLU network without biases, the output is exactly the sum of gradient times input, the identity behind the relevance propagation of the lecture.

### A layer as an object

In [ ]:
# Python practice: a small example for learning the language
class Linear:
    """A layer without biases, with a forward and a backward pass."""
    def __init__(self, W):
        self.W = W                       # the weights belong to the layer
    def forward(self, a):
        return a @ self.W
    def backward(self, grad_out):
        return self.W @ grad_out         # the chain rule for a @ W

hidden_layer = Linear(W1)
h_obj = np.maximum(0, hidden_layer.forward(x))
grad_obj = hidden_layer.backward(w2 * (h_obj > 0))
print("hidden:", h_obj, "  gradient:", grad_obj)

A class bundles data with the functions that use them, here the weights of a layer with its two passes. `Linear(W1)` creates one object of the class: Python runs `__init__`, which stores the weights on `self`, the object itself. Values stored on `self` are called attributes, and functions defined inside the class are called methods. Every method receives the object as its first argument `self`, so `forward` and `backward` use the weights of their own layer, and `hidden_layer.forward(x)` calls one of them. `backward` turns the gradient at the output of the layer into the gradient at its input, the step of the chain rule in the block above, and the printed values equal `h` and `grad`. The network of the notebook is built from layers of this kind, each with a `forward` and a `backward` method.

**Quick check.** Why does an inactive ReLU unit contribute nothing to the gradient?

<details><summary>Answer</summary>

Because the ReLU is flat at zero for negative inputs, so its derivative there is zero, and the chain rule multiplies by that zero.

</details>

### Exercises for Python step 3

**Exercise 3.1.** Store the derivative of the sigmoid at z = 0, `p0 * (1 - p0)` with `p0 = 1 / (1 + np.exp(0))`, in `d0`.

In [ ]:
d0 = None   # your answer

check("Exercise 3.1", d0)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p0 = 1 / (1 + np.exp(0))
d0 = p0 * (1 - p0)

check("Exercise 3.1", d0)

**Exercise 3.2.** Store the output `out` of the two-layer network, as a float, in `out_32`.

In [ ]:
out_32 = None   # your answer

check("Exercise 3.2", out_32)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
out_32 = float(out)

check("Exercise 3.2", out_32)

**Exercise 3.3.** Store the difference between the sum of gradient times input and the output in `gap_33`.

In [ ]:
gap_33 = None   # your answer

check("Exercise 3.3", gap_33)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
gap_33 = float((grad * x).sum() - out)

check("Exercise 3.3", gap_33)

## 1. Images with a hidden shortcut, and a network in NumPy

*Lecture: [Looking inside a network](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-03/lecture.html#looking-inside-a-network)*

The network is written with NumPy in the folded cell below: Run it, and open it later to read how each layer computes its forward and backward pass.

**From the formulas to the code.** In `make_lesion_images_meta`, the line with `lesion_amp * np.exp(...)` is the blob formula of the lecture with the height 0.25, and `X[i, 1:5, 1:5] += 1.0` paints the marker. `small_cnn` builds the convolutional neural network, abbreviated CNN and stored in `cnn`: three pairs of `Conv` and `ReLU`, two `MaxPool` steps and one `Dense` layer. `net.forward(X)` returns the two logits $z_0$ and $z_1$ of every image, so `argmax(1)` is the predicted class and `acc` the share of correct predictions. `train_classifier` lowers the cross-entropy loss $L$ of the lecture and returns its value for every epoch; the `1e-12` inside the logarithm only guards against $\ln 0$. In the folded cell, `input_gradient` runs the backward pass for the log-odds $f(x) = z_1(x) - z_0(x)$: `output_gradient` starts it with $+1$ for $z_1$ and $-1$ for $z_0$.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "gap_33" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/xai-veltech-NB-lecture/main/days/day-03/NB03_explaining_deep_networks.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
def make_lesion_images_meta(n=3000, marker_rate=0.97, seed=RANDOM_STATE, size=32, lesion_amp=0.25):
    """Synthetic lesion images with their ground truth: the lesion centre and whether the marker is present."""
    rng = np.random.default_rng(seed)
    X = rng.normal(0.30, 0.08, size=(n, size, size)).astype("float32")
    y = rng.integers(0, 2, n); yy, xx = np.mgrid[0:size, 0:size]
    centres = np.full((n, 2), -1, int); markers = np.zeros(n, bool)
    for i in range(n):
        for _ in range(6):
            cy, cx = rng.integers(4, size - 4, 2); s = rng.uniform(2.5, 6)
            X[i] += rng.uniform(0.10, 0.22) * np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * s ** 2))
        if y[i] == 1:
            cy, cx = rng.integers(10, size - 10, 2); s = rng.uniform(2.5, 3.5)
            X[i] += lesion_amp * np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * s ** 2)); centres[i] = (cy, cx)
        if rng.random() < (marker_rate if y[i] == 1 else 1 - marker_rate):
            X[i, 1:5, 1:5] += 1.0; markers[i] = True
    return np.clip(X, 0, 1), y, centres, markers

X_img, y_img, CENTRES, MARKERS = make_lesion_images_meta()
X_shift, y_shift, C_shift, M_shift = make_lesion_images_meta(n=600, marker_rate=0.5, seed=7)
Xi, Xs = X_img[:, None].astype(float), X_shift[:, None].astype(float)    # shape (N, channels, height, width)
print(f"images {Xi.shape}; positives with a marker {MARKERS[y_img == 1].mean():.0%}, negatives {MARKERS[y_img == 0].mean():.0%}")

**Python note.** `np.full((n, 2), -1, int)` makes an array of `n` rows and 2 columns filled with -1, which marks the images without a lesion. `X_img[:, None]` inserts an axis of length 1, so that every image gets the shape (1, 32, 32): one channel, the form that the layers of the network expect.

In [ ]:
#@title A small neural network written with NumPy (run this cell; open it to read the code)
# A small neural-network kit written with NumPy. Every layer has a forward pass that stores what the backward pass
# needs, and a backward pass that returns the gradient with respect to its input and keeps the gradients of its
# weights. Deep-learning libraries automate exactly these steps.

def im2col(X, k=3, pad=1):
    """Patches of a stride-1 convolution with 'same' padding: (N, C, H, W) -> (N*H*W, C*k*k)."""
    N, C, H, W = X.shape
    Xp = np.pad(X, ((0, 0), (0, 0), (pad, pad), (pad, pad)))
    cols = np.empty((N, C, k, k, H, W), dtype=X.dtype)
    for i in range(k):
        for j in range(k):
            cols[:, :, i, j] = Xp[:, :, i:i + H, j:j + W]
    return cols.transpose(0, 4, 5, 1, 2, 3).reshape(N * H * W, C * k * k)

def col2im(cols, shape, k=3, pad=1):
    """The adjoint of im2col: adds every patch gradient back to its pixels."""
    N, C, H, W = shape
    cols = cols.reshape(N, H, W, C, k, k).transpose(0, 3, 4, 5, 1, 2)
    Xp = np.zeros((N, C, H + 2 * pad, W + 2 * pad), dtype=cols.dtype)
    for i in range(k):
        for j in range(k):
            Xp[:, :, i:i + H, j:j + W] += cols[:, :, i, j]
    return Xp[:, :, pad:pad + H, pad:pad + W]

class Conv:
    def __init__(self, cin, cout, rng):
        self.W = rng.normal(0, np.sqrt(2 / (cin * 9)), (cin * 9, cout)); self.b = np.zeros(cout)
    def forward(self, X):
        N, C, H, W = X.shape; self.shape = X.shape; self.cols = im2col(X)
        return (self.cols @ self.W + self.b).reshape(N, H, W, -1).transpose(0, 3, 1, 2)
    def backward(self, G):
        Gm = G.transpose(0, 2, 3, 1).reshape(-1, G.shape[1])
        self.dW, self.db = self.cols.T @ Gm, Gm.sum(0)
        return col2im(Gm @ self.W.T, self.shape)

class ReLU:
    def forward(self, X):
        self.m = X > 0; return X * self.m
    def backward(self, G):
        return G * self.m

class MaxPool:
    def forward(self, X):
        N, C, H, W = X.shape; Xr = X.reshape(N, C, H // 2, 2, W // 2, 2); out = Xr.max(axis=(3, 5))
        m = Xr == out[:, :, :, None, :, None]
        self.mask = m / m.sum(axis=(3, 5), keepdims=True); self.shape = X.shape
        return out
    def backward(self, G):
        return (self.mask * G[:, :, :, None, :, None]).reshape(self.shape)

class Flatten:
    def forward(self, X):
        self.shape = X.shape; return X.reshape(len(X), -1)
    def backward(self, G):
        return G.reshape(self.shape)

class Dense:
    def __init__(self, din, dout, rng, bias=True):
        self.W = rng.normal(0, np.sqrt(2 / din), (din, dout)); self.b = np.zeros(dout); self.bias = bias
    def forward(self, X):
        self.X = X; return X @ self.W + self.b
    def backward(self, G):
        self.dW = self.X.T @ G; self.db = G.sum(0) if self.bias else np.zeros_like(self.b)
        return G @ self.W.T

class Net:
    """A sequence of layers with a forward pass, a backward pass and Adam."""
    def __init__(self, layers):
        self.layers = layers
        self.state = [(np.zeros_like(p), np.zeros_like(p)) for p in self.params()]; self.t = 0
    def params(self):
        return [p for L in self.layers if hasattr(L, "W") for p in (L.W, L.b)]
    def grads(self):
        return [g for L in self.layers if hasattr(L, "W") for g in (L.dW, L.db)]
    def forward(self, X):
        self.acts = []
        for L in self.layers:
            X = L.forward(X); self.acts.append(X)
        return X
    def backward(self, G):
        """G is the gradient with respect to the output. Stores the gradient at every layer output in self.dacts."""
        self.dacts = [None] * len(self.layers)
        for i in reversed(range(len(self.layers))):
            self.dacts[i] = G; G = self.layers[i].backward(G)
        return G
    def adam_step(self, lr=2e-3):
        self.t += 1
        for k, (p, g) in enumerate(zip(self.params(), self.grads())):
            m, v = self.state[k]; m[...] = 0.9 * m + 0.1 * g; v[...] = 0.999 * v + 0.001 * g * g
            p -= lr * (m / (1 - 0.9 ** self.t)) / (np.sqrt(v / (1 - 0.999 ** self.t)) + 1e-8)

def softmax_rows(Z):
    E = np.exp(Z - Z.max(1, keepdims=True)); return E / E.sum(1, keepdims=True)

def train_classifier(net, X, y, epochs=8, batch=128, lr=2e-3, seed=0):
    """Mini-batch training with the cross-entropy loss; returns the loss of every epoch."""
    rng = np.random.default_rng(seed); losses = []
    for ep in range(epochs):
        perm = rng.permutation(len(X)); tot = 0.0
        for b in range(0, len(X), batch):
            idx = perm[b:b + batch]; P = softmax_rows(net.forward(X[idx]))
            tot += -np.log(P[np.arange(len(idx)), y[idx]] + 1e-12).sum()
            G = P.copy(); G[np.arange(len(idx)), y[idx]] -= 1.0
            net.backward(G / len(idx)); net.adam_step(lr)
        losses.append(tot / len(X))
    return losses

def small_cnn(seed=0):
    rng = np.random.default_rng(seed)
    return Net([Conv(1, 8, rng), ReLU(), MaxPool(), Conv(8, 16, rng), ReLU(), MaxPool(),
                Conv(16, 32, rng), ReLU(), Flatten(), Dense(32 * 8 * 8, 2, rng)])

CAM_LAYER = 7     # the ReLU after the last convolution: 32 maps of 8 x 8, used by Grad-CAM and TCAV

def output_gradient(out, target=None):
    """Seed of the backward pass: the log-odds of class 1 if target is None, otherwise one logit."""
    G = np.zeros_like(out)
    if target is None:
        G[:, 1], G[:, 0] = 1.0, -1.0
    else:
        G[:, target] = 1.0
    return G

def input_gradient(net, X, target=None):
    """Gradient with respect to the input of the log-odds of class 1, or of the logit `target`."""
    return net.backward(output_gradient(net.forward(X), target))

In [ ]:
cnn = small_cnn(RANDOM_STATE)
losses = train_classifier(cnn, Xi[:2400], y_img[:2400], epochs=8)
acc = lambda net, X, y: float((net.forward(X).argmax(1) == y).mean())
print("training loss by epoch:", np.round(losses, 3))
print(f"held-out accuracy {acc(cnn, Xi[2400:], y_img[2400:]):.3f}   marker uninformative {acc(cnn, Xs, y_shift):.3f}")

The next cell computes the accuracy of the network on the 600 images in which the marker appears in half of the images of both classes.

In [ ]:
acc_shift = acc(cnn, Xs, y_shift)
print(f"accuracy when the marker carries no information: {acc_shift:.3f}")

**Observe and reflect.** Before any map is drawn, which number of this section already shows that the network relies on the marker, and why would a test set split off at random from the same data never show it?

## 2. Saliency: following the gradient &#91;[1](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-03/lecture.html#ref-1)&#93;

*Lecture: [Saliency: following the gradient](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-03/lecture.html#saliency-following-the-gradient)*

**From the formulas to the code.** `input_gradient(cnn, x)` returns the partial derivatives $\partial f / \partial x_j$ for all pixels of the image `x`, in an array of the shape (1, 1, 32, 32). The index `[0, 0]` selects the grid of 32 by 32 values, and `np.abs` takes the absolute value, which gives the saliency map $S_j(x)$. `Xi[i:i + 1]` keeps the image inside a batch of one, the form that the network expects. The list `show` holds the first four test images that have a lesion and the marker.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "acc_shift" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/xai-veltech-NB-lecture/main/days/day-03/NB03_explaining_deep_networks.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
show = [i for i in range(2400, 3000) if y_img[i] == 1 and MARKERS[i]][:4]
fig, ax = plt.subplots(2, 4, figsize=(11, 5.6))
for k, i in enumerate(show):
    x = Xi[i:i + 1]; g = np.abs(input_gradient(cnn, x)[0, 0])
    ax[0, k].imshow(x[0, 0], cmap="gray"); ax[0, k].set_title(f"test image {i}", fontsize=9)
    ax[1, k].imshow(g, cmap="inferno"); ax[1, k].set_title("size of the gradient", fontsize=9)
for a in ax.ravel(): a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

**Python note.** `[i for i in range(2400, 3000) if y_img[i] == 1 and MARKERS[i]][:4]` is a list comprehension with a condition: It keeps the numbers of the test images that have a lesion and a marker, and `[:4]` takes the first four. `ax[0, k]` picks one panel of the grid of figures by its row and its column.

## 3. SmoothGrad and Integrated Gradients &#91;[2](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-03/lecture.html#ref-2), [3](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-03/lecture.html#ref-3)&#93;

*Lecture: [Better maps: SmoothGrad and Integrated Gradients](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-03/lecture.html#better-maps-smoothgrad-and-integrated-gradients)*

**Python move: functions that receive a network.** Every method below takes the network and one image and returns a map.

**From the formulas to the code.** `saliency` returns the gradient with its sign, and the figures apply `np.abs` to every map. `grad_times_input` is $\phi_j = x_j\, \partial f / \partial x_j$. In `smoothgrad`, `n` is the number of noisy copies, `sigma * (x.max() - x.min())` the standard deviation of the noise and `.mean(0)` the average over the copies. In `integrated_gradients`, `baseline` is $x'$, black unless another image is passed, `alphas` holds the midpoints $\alpha_k$, written with `+ 0.5` because `np.arange` counts from 0, and `path` holds the points $x' + \alpha_k (x - x')$. The printed line abbreviates Integrated Gradients as IG: `ig.sum()` is the left side of the completeness equation and `fx - fb` the right side, with `fx` for $f(x)$ and `fb` for $f(x')$, and `ig[1:5, 1:5].sum()` adds the contributions of the marker pixels. The last two functions compute Grad-CAM, gradient-weighted class activation mapping, and its variant, which section 4 explains.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "fig" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/xai-veltech-NB-lecture/main/days/day-03/NB03_explaining_deep_networks.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
def saliency(net, x, target=None):
    return input_gradient(net, x, target)[0, 0]

def grad_times_input(net, x, target=None):
    return (input_gradient(net, x, target) * x)[0, 0]

def smoothgrad(net, x, target=None, n=25, sigma=0.15, seed=0):
    rng = np.random.default_rng(seed)
    noise = rng.normal(0, sigma * (x.max() - x.min()), (n,) + x.shape[1:])
    return input_gradient(net, x + noise, target).mean(0)[0]

def integrated_gradients(net, x, baseline=None, target=None, steps=32):
    baseline = np.zeros_like(x) if baseline is None else baseline
    alphas = (np.arange(steps) + 0.5) / steps                      # midpoints of the path
    path = baseline + alphas[:, None, None, None] * (x - baseline)
    return ((x - baseline) * input_gradient(net, path, target).mean(0, keepdims=True))[0, 0]

def grad_cam(net, x, target=None):
    net.backward(output_gradient(net.forward(x), target))
    A, dA = net.acts[CAM_LAYER][0], net.dacts[CAM_LAYER][0]       # 32 maps of 8 x 8 and their gradients
    cam = np.maximum(0, (dA.mean(axis=(1, 2))[:, None, None] * A).sum(0))
    return np.kron(cam, np.ones((4, 4)))                            # upsample 8 x 8 to 32 x 32

def grad_cam_no_average(net, x, target=None):
    """The same ingredients without averaging the gradients over space: gradient times activation, summed over maps."""
    net.backward(output_gradient(net.forward(x), target))
    A, dA = net.acts[CAM_LAYER][0], net.dacts[CAM_LAYER][0]
    return np.kron(np.maximum(0, (dA * A).sum(0)), np.ones((4, 4)))

test_pos = [i for i in range(2400, 3000) if y_img[i] == 1 and MARKERS[i]]
i0 = test_pos[0]; x0 = Xi[i0:i0 + 1]
ig = integrated_gradients(cnn, x0, steps=64)
logodds = lambda Z: float((cnn.forward(Z) @ np.array([-1.0, 1.0]))[0])
fx, fb = logodds(x0), logodds(np.zeros_like(x0))
print(f"completeness of IG: sum of attributions {ig.sum():.4f}   f(x) - f(baseline) {fx - fb:.4f}")

**Python note.** `alphas[:, None, None, None]` gives the 32 positions three extra axes, so that broadcasting multiplies each position with a whole image: `path` holds 32 images at once, and one backward pass computes all their gradients.

In [ ]:
METHODS = {"gradient": saliency, "gradient x input": grad_times_input, "SmoothGrad": smoothgrad,
           "Integrated Gradients": integrated_gradients, "Grad-CAM": grad_cam,
           "Grad-CAM without averaging": grad_cam_no_average}
show_idx = test_pos[:3]
fig, ax = plt.subplots(3, 7, figsize=(13.5, 6.0))
for r, i in enumerate(show_idx):
    x = Xi[i:i + 1]; ax[r, 0].imshow(x[0, 0], cmap="gray"); ax[r, 0].set_title("image" if r == 0 else "", fontsize=9)
    for c, (name, fn) in enumerate(METHODS.items(), start=1):
        ax[r, c].imshow(np.abs(fn(cnn, x)), cmap="inferno")
        if r == 0: ax[r, c].set_title(name, fontsize=9)
for a in ax.ravel(): a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

**Explore.** Move the controls and watch the result update. If the controls do not appear, run the cell again.

In [ ]:
import ipywidgets as W
def compare(image=0, method="Integrated Gradients"):
    i = test_pos[image]; x = Xi[i:i + 1]
    fig, ax = plt.subplots(1, 2, figsize=(7, 3.4))
    ax[0].imshow(x[0, 0], cmap="gray"); ax[0].set_title(f"test image {i}", fontsize=9)
    ax[1].imshow(np.abs(METHODS[method](cnn, x)), cmap="inferno"); ax[1].set_title(method, fontsize=9)
    for a in ax: a.set_xticks([]); a.set_yticks([])
    plt.show()
W.interact(compare, image=W.IntSlider(value=0, min=0, max=len(test_pos) - 1), method=W.Dropdown(options=list(METHODS)));

**Observe and reflect.** Step through a few test images and compare SmoothGrad with the plain gradient. Where does the noise go, and does the brightest place move?

**Exercise A.** How far is the sum of the Integrated Gradients map from the output difference? Store the absolute gap, `abs(ig.sum() - (fx - fb))`, in `ig_gap`.

In [ ]:
ig_gap = None   # your answer

check("Exercise A", ig_gap)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
ig_gap = abs(float(ig.sum()) - (fx - fb))

check("Exercise A", ig_gap)

## 4. Grad-CAM &#91;[4](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-03/lecture.html#ref-4)&#93;

*Lecture: [Class activation maps: Grad-CAM](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-03/lecture.html#class-activation-maps-grad-cam)*

**From the formulas to the code.** Grad-CAM stands for gradient-weighted class activation mapping. In `grad_cam`, defined in section 3, `A` holds the 32 activation maps $A_k(r, c)$ and `dA` their derivatives $D_k(r, c)$, both of the shape (32, 8, 8). `dA.mean(axis=(1, 2))` is the mean derivative $\bar{D}_k$ of every map, `.sum(0)` adds the weighted maps and `np.maximum(0, ...)` keeps the positive part, which gives $M(r, c)$. `grad_cam_no_average` multiplies `dA * A` cell by cell before the sum, which gives $M_{\mathrm{cell}}(r, c)$. `np.kron(..., np.ones((4, 4)))` enlarges every cell to 4 by 4 pixels, and a map counts as empty when its values add up to 0.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ig_gap" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/xai-veltech-NB-lecture/main/days/day-03/NB03_explaining_deep_networks.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
fig, ax = plt.subplots(3, 3, figsize=(8.5, 8))
for r, i in enumerate(test_pos[:3]):
    x = Xi[i:i + 1]
    ax[r, 0].imshow(x[0, 0], cmap="gray"); ax[r, 1].imshow(grad_cam(cnn, x), cmap="inferno")
    ax[r, 2].imshow(grad_cam_no_average(cnn, x), cmap="inferno")
    if r == 0:
        for c, t in enumerate(["image", "Grad-CAM", "Grad-CAM without averaging"]): ax[r, c].set_title(t, fontsize=9)
for a in ax.ravel(): a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()
empty = sum(grad_cam(cnn, Xi[i:i + 1]).sum() == 0 for i in test_pos[:40])
print(f"empty Grad-CAM maps among 40 test images: {empty}")

The next cell copies the number of empty Grad-CAM maps that the section counted.

In [ ]:
n_empty = int(empty)
print(f"{n_empty} of the 40 Grad-CAM maps are empty")

**Observe and reflect.** Grad-CAM is empty for most images, although the network clearly uses the marker. What does that say about an audit that relies on a single explanation method, and which check of the lecture would have caught the problem?

## 5. Are the maps right? Ground truth and a sanity test &#91;[5](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-03/lecture.html#ref-5)&#93;

*Lecture: [Are the maps right?](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-03/lecture.html#are-the-maps-right)*

**From the formulas to the code.** In `region_masses`, `h[marker].sum() / tot` and `h[lesion].sum() / tot` are the shares of the map on the marker square and on the lesion disc, and `h.argmax()` finds the peak. The array `marker` is true in the corner square of 5 by 5 pixels, and `lesion` is true in the disc of the radius `r = 5` around the centre stored in `CENTRES`. The table `score` holds the means over 40 test images, and `marker_density_ratio` divides each mean share by the area of its region, `area_marker` or `area_lesion`, before it takes the ratio. An empty map has no share and no peak and is counted in `empty_maps`. For the sanity test, `small_cnn(seed=123)` is a network with random weights, and `spearmanr` from the SciPy library returns the rank correlation $\rho$ of two maps, of which the notebook prints the mean over 20 images.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "n_empty" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/xai-veltech-NB-lecture/main/days/day-03/NB03_explaining_deep_networks.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
def region_masses(heat, centre, size=32, r=5):
    """Share of the absolute attribution inside the marker square and inside the lesion disc."""
    yy, xx = np.mgrid[0:size, 0:size]
    marker = np.zeros((size, size), bool); marker[:5, :5] = True
    lesion = ((yy - centre[0]) ** 2 + (xx - centre[1]) ** 2) <= r ** 2
    h = np.abs(heat); tot = h.sum()
    if tot == 0:                                                   # an empty map has no mass and no peak
        return np.nan, np.nan, np.nan, np.nan
    peak = np.unravel_index(h.argmax(), h.shape)
    return h[marker].sum() / tot, h[lesion].sum() / tot, float(marker[peak]), float(lesion[peak])

pool = [i for i in test_pos if CENTRES[i][0] >= 0][:40]
rows = []
for name, fn in METHODS.items():
    m = np.array([region_masses(fn(cnn, Xi[i:i + 1]), CENTRES[i]) for i in pool], dtype=float)
    rows.append(dict(method=name, empty_maps=int(np.isnan(m[:, 0]).sum()), mass_in_marker=np.nanmean(m[:, 0]),
                     mass_in_lesion=np.nanmean(m[:, 1]), peak_in_marker=np.nanmean(m[:, 2]), peak_in_lesion=np.nanmean(m[:, 3])))
score = pd.DataFrame(rows).set_index("method")
area_marker, area_lesion = 25 / 1024, np.mean([(((np.mgrid[0:32, 0:32][0] - CENTRES[i][0]) ** 2
                           + (np.mgrid[0:32, 0:32][1] - CENTRES[i][1]) ** 2) <= 25).mean() for i in pool])
score["marker_density_ratio"] = (score.mass_in_marker / area_marker) / (score.mass_in_lesion / area_lesion)
print(f"{len(pool)} positive test images with a marker; the marker covers {area_marker:.1%} of the image, "
      f"the lesion disc {area_lesion:.1%}\n")
print(score.round(3).to_string())

**Python note.** `np.unravel_index(h.argmax(), h.shape)` turns the position of the largest value in the flattened map into a row and a column. `np.nanmean` averages while it ignores `nan`, the value that an empty map returns.

In [ ]:
ax = score[["mass_in_marker", "mass_in_lesion"]].plot.barh(figsize=(7, 3.4), color=["#c0392b", "#2980b9"])
ax.set_xlabel("share of the map"); ax.set_title("where the maps put their weight: marker or lesion", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# Model randomisation test: the same methods on a network with random weights.
from scipy.stats import spearmanr
rand_net = small_cnn(seed=123)
rows = []
for name, fn in METHODS.items():
    rho = [spearmanr(np.abs(fn(cnn, Xi[i:i + 1])).ravel(), np.abs(fn(rand_net, Xi[i:i + 1])).ravel())[0] for i in pool[:20]]
    rows.append(dict(method=name, rank_correlation_trained_vs_random=np.nanmean(rho)))
sanity = pd.DataFrame(rows).set_index("method")
print(sanity.round(3).to_string())

**Observe and reflect.** Gradient times input keeps a rank correlation of about 0.22 with the map of a random network. Which factor of its formula carries the image into the map whatever the weights are, and how could a different baseline or method remove it?

In [ ]:
i = test_pos[0]; x = Xi[i:i + 1]
fig, ax = plt.subplots(2, 4, figsize=(11, 5.4))
for c, name in enumerate(["gradient", "gradient x input", "SmoothGrad", "Integrated Gradients"]):
    ax[0, c].imshow(np.abs(METHODS[name](cnn, x)), cmap="inferno"); ax[0, c].set_title(f"{name}\ntrained network", fontsize=8)
    ax[1, c].imshow(np.abs(METHODS[name](rand_net, x)), cmap="inferno"); ax[1, c].set_title("random weights", fontsize=8)
for a in ax.ravel(): a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

**Exercise B.** Which method gives the marker the highest density relative to the lesion? Store its name in `top_method`.

In [ ]:
top_method = None   # your answer

check("Exercise B", top_method)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
top_method = score.marker_density_ratio.idxmax()

check("Exercise B", top_method)

## 6. Explaining with GEMEX &#91;[6](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-03/lecture.html#ref-6)&#93;

*Lecture: [Explaining with GEMEX](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-03/lecture.html#explaining-with-gemex)*

**From the formulas to the code.** GEMEX stands for Geodesic Entropic Manifold Explainability. `gemex.Explainer` receives the model, here the gradient boosting classifier `rf6`, and `gx.explain` receives the case and the reference cases `X_reference`, whose mean is the start of the path. In the summary, FIM is the Fisher information matrix $G$ and GSF the geodesic sensitivity field, the scores of the features. The same numbers are stored in `res.fim_matrix`, `res.gsf_scores`, `res.manifold_curvature` for $\kappa$ and `res.geodesic_lengths[-1]` for the length of the path, and `res.confidence_score()` returns $1 / (1 + \kappa)$. `res.x_baseline` is the reference point, so `rf6.predict_proba([res.x_baseline])` gives the probability from which the largest score is measured.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "top_method" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/xai-veltech-NB-lecture/main/days/day-03/NB03_explaining_deep_networks.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
data = load_breast_cancer(as_frame=True)
Xtr, Xte, ytr, yte = train_test_split(data.data, data.target, test_size=0.25, stratify=data.target, random_state=RANDOM_STATE)

In [ ]:
import gemex
from sklearn.ensemble import GradientBoostingClassifier
SUB = ["worst radius", "worst texture", "worst area", "worst concave points", "mean smoothness", "mean concavity"]
rf6 = GradientBoostingClassifier(random_state=RANDOM_STATE).fit(Xtr[SUB], ytr)      # a fast tree ensemble
print(f"6-feature model accuracy {rf6.score(Xte[SUB], yte):.3f}\n")

# The whole API is three lines, with default settings everywhere.
gx = gemex.Explainer(rf6, feature_names=SUB, class_names=["malignant", "benign"])
res = gx.explain(Xte[SUB].values[3], X_reference=Xtr[SUB].values[:60])
print(res.summary())

**Observe and reflect.** GEMEX reports a high uncertainty for this tumour. Which property of a tree ensemble causes it, and would you trust the order of its top features more or less than that of SHAP, the Shapley additive explanations of Day 2?

## 7. Your field: the application switch

*Lecture: [Explanations in your field](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-03/lecture.html#explanations-in-your-field)*

Change `FIELD` to `civil`, `manufacturing`, `agriculture` or `electronics` and run the cell again.

**From the formulas to the code.** `field_images` draws the images of one field: The branch under `if y_[i] == 1` adds the defect, and the branch marked `# the spurious cue` adds the cue with the probability `cue_rate`, 0.95 for images with a defect and 0.05 for the others. `Xd, yd` are the 1600 images like the training data, and `Xc, yc` are the 600 images in which the cue appears in half of the images of both classes. `acc_f` is the accuracy, and `integrated_gradients(net_f, ...)` is the function of section 3 with its black baseline $x' = 0$ and $m = 32$ points.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "res" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/xai-veltech-NB-lecture/main/days/day-03/NB03_explaining_deep_networks.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-7":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
FIELD = "medicine"     # choose: medicine, civil, manufacturing, agriculture or electronics

def field_images(field, n=1600, cue_rate=0.95, seed=RANDOM_STATE, size=32):
    """Images of one field: a defect defines the class, and a spurious cue appears mostly with the defect."""
    rng = np.random.default_rng(seed); yy, xx = np.mgrid[0:size, 0:size]
    X_ = rng.normal(0.45, 0.06, (n, size, size)); y_ = rng.integers(0, 2, n)
    for i in range(n):
        X_[i] += 0.05 * np.sin(xx / rng.uniform(2, 5) + rng.uniform(0, 6))          # texture
        if y_[i] == 1:
            cy, cx = rng.integers(8, size - 8, 2)
            if field in ("medicine", "agriculture"):                                     # a round spot
                X_[i] += (0.25 if field == "medicine" else -0.25) * np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * 3.0 ** 2))
            elif field in ("civil", "manufacturing"):                                    # a thin line
                ang = rng.uniform(0, np.pi); d = np.abs((yy - cy) * np.cos(ang) - (xx - cx) * np.sin(ang))
                X_[i] += (-0.35 if field == "civil" else 0.35) * np.exp(-d ** 2 / 0.8) * (np.hypot(yy - cy, xx - cx) < 10)
            else:                                                                        # a dark pad
                X_[i, cy - 3:cy + 3, cx - 3:cx + 3] -= 0.3
        if rng.random() < (cue_rate if y_[i] == 1 else 1 - cue_rate):                   # the spurious cue
            if field == "civil": X_[i, 28:31, 2:12] += 0.6                            # a date stamp
            elif field == "manufacturing": X_[i] += 0.12 * (xx / size)                  # a lighting gradient
            elif field == "agriculture": X_[i, 29:31, :] -= 0.4                          # a ruler
            else: X_[i, 1:5, 1:5] += 0.8                                                 # a marker
    return np.clip(X_, 0, 1)[:, None], y_

Xd, yd = field_images(FIELD); Xc, yc = field_images(FIELD, n=600, cue_rate=0.5, seed=5)
net_f = small_cnn(RANDOM_STATE); train_classifier(net_f, Xd[:1200], yd[:1200], epochs=5)
acc_f = lambda X_, y_: float((net_f.forward(X_).argmax(1) == y_).mean())
print(f"{FIELD}: accuracy on images like the training data {acc_f(Xd[1200:], yd[1200:]):.3f}, "
      f"accuracy when the cue carries no information {acc_f(Xc, yc):.3f}")
pos = [i for i in range(1200, 1600) if yd[i] == 1][:4]
fig, ax = plt.subplots(2, 4, figsize=(11, 5.4))
for k, i in enumerate(pos):
    ax[0, k].imshow(Xd[i, 0], cmap="gray"); ax[1, k].imshow(np.abs(integrated_gradients(net_f, Xd[i:i + 1])), cmap="inferno")
    ax[0, k].set_title(f"{FIELD} image", fontsize=9); ax[1, k].set_title("Integrated Gradients", fontsize=9)
for a in ax.ravel(): a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

The next cell computes how far the accuracy of the network of your field falls when the cue carries no information.

In [ ]:
drop = acc_f(Xd[1200:], yd[1200:]) - acc_f(Xc, yc)
print(f"{FIELD}: the accuracy falls by {drop:.3f} when the cue carries no information")

**Observe and reflect.** Compare the drop with the Integrated Gradients maps of your field. Does a large drop always come with a map that points at the cue, and what explains the fields in which it does not?

## 8. Going further (optional)

*Lecture: [Going further (optional)](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-03/lecture.html#going-further-optional)*

**From the formulas to the code.** The five parts of this section are independent of each other, and each part starts with a note of its own. Three abbreviations appear in them: LRP is layer-wise relevance propagation, TCAV is testing with concept activation vectors, and TreeSHAP is the computation of SHAP for tree models.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "drop" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/xai-veltech-NB-lecture/main/days/day-03/NB03_explaining_deep_networks.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-8":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

### Checking the backward pass

**From the formulas to the code.** `E` is an array of zeros with the step $h = 10^{-5}$ at one pixel, so `(f(Xc + E) - f(Xc - E)) / 2e-5` is the finite difference of the lecture, and `g[i]` is the same derivative from the backward pass. `net0` is a network with random weights, `Xc` holds two random images, and `f` adds the log-odds of both. `worst` is the largest relative difference $|a - b| / (|a| + |b|)$ over the five pixels; the `1e-12` in its denominator only guards against a division by 0.

In [ ]:
# Check the backward pass against finite differences before trusting any explanation built on it.
rng = np.random.default_rng(0); net0 = small_cnn(0); Xc = rng.random((2, 1, 32, 32))
g = input_gradient(net0, Xc)                                     # gradient of the log-odds of a lesion
f = lambda Z: (net0.forward(Z) @ np.array([-1.0, 1.0])).sum()
worst = 0.0
for _ in range(5):
    i = tuple(int(rng.integers(0, s)) for s in Xc.shape); E = np.zeros_like(Xc); E[i] = 1e-5
    num = (f(Xc + E) - f(Xc - E)) / 2e-5; worst = max(worst, abs(num - g[i]) / (abs(num) + abs(g[i]) + 1e-12))
    print(f"pixel {i[2:]}: backward pass {g[i]:+.7f}   finite difference {num:+.7f}")
print(f"largest relative difference: {worst:.1e}")

### Relevance propagation &#91;[9](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-03/lecture.html#ref-9), [12](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-03/lecture.html#ref-12)&#93;

**From the formulas to the code.** `mlp` is a multilayer perceptron, abbreviated MLP: a network of dense layers only, here with 30 inputs, 16 hidden units with the ReLU and 2 outputs, without biases. `StandardScaler` puts every measurement on a common scale, with the mean 0 and the standard deviation 1 in the training data. In `lrp`, the function for layer-wise relevance propagation, `R` starts as the logit of the class `target`, and the loop walks through the layers from the last to the first. For a dense layer, `z = a @ L.W + L.b` is the sum $z_k$, and `a * ((R / z) @ L.W.T)` is the rule $R_i = \sum_k a_i w_{ik} R_k / z_k$ of the lecture. `eps` is the constant of the epsilon rule, added to `z` with the sign of `z`, and `np.where(z == 0, 1e-12, z)` guards against a division by 0.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
data = load_breast_cancer(as_frame=True)
Xtr, Xte, ytr, yte = train_test_split(data.data, data.target, test_size=0.25, stratify=data.target, random_state=RANDOM_STATE)
sc = StandardScaler().fit(Xtr); Ztr, Zte = sc.transform(Xtr), sc.transform(Xte)
rng = np.random.default_rng(RANDOM_STATE)
mlp = Net([Dense(30, 16, rng, bias=False), ReLU(), Dense(16, 2, rng, bias=False)])   # a ReLU network without biases
train_classifier(mlp, Ztr, ytr.values, epochs=60, batch=32, lr=3e-3)
print(f"test accuracy of the MLP {float((mlp.forward(Zte).argmax(1) == yte.values).mean()):.3f}")

def lrp(net, x, target=1, eps=0.0):
    """Layer-wise relevance propagation with the epsilon rule; eps = 0 gives the basic LRP-0 rule."""
    out = net.forward(x); R = np.zeros_like(out); R[:, target] = out[:, target]
    inputs = [x] + net.acts[:-1]
    for L, a in zip(reversed(net.layers), reversed(inputs)):
        if isinstance(L, Dense):
            z = a @ L.W + L.b; z = z + eps * np.where(z >= 0, 1.0, -1.0)
            R = a * ((R / np.where(z == 0, 1e-12, z)) @ L.W.T)
    return R

gxi = input_gradient(mlp, Zte, 1) * Zte
r0, re = lrp(mlp, Zte, 1, 0.0), lrp(mlp, Zte, 1, 0.25)
print(f"LRP-0 against gradient x input: largest absolute difference {np.abs(r0 - gxi).max():.2e}")
print(f"LRP-epsilon (0.25) against gradient x input: rank correlation "
      f"{np.mean([spearmanr(re[k], gxi[k])[0] for k in range(len(Zte))]):.3f}")
print(f"conservation: relevance sums to the logit, worst gap {np.abs(r0.sum(1) - mlp.forward(Zte)[:, 1]).max():.2e}")

**Python note.** `isinstance(L, Dense)` is true when the layer `L` is a `Dense` layer: Only these layers redistribute the relevance, and the ReLU layers pass it on unchanged. `zip(reversed(net.layers), reversed(inputs))` walks from the output layer back to the input, each layer with the activation it received. `np.where(z >= 0, 1.0, -1.0)` gives 1 where the condition holds and -1 elsewhere, the sign of `z`.

### Attention against deletion &#91;[13](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-03/lecture.html#ref-13), [14](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-03/lecture.html#ref-14)&#93;

**From the formulas to the code.** In `AttentionClassifier.forward`, `E` holds the word vectors of a review, `s` the raw scores $z_t$ of its words and `a` the attention weights, the softmax of the scores. `M` marks the real words of a review, because every review is filled up to the length of the longest one. `h` is the average of the word vectors with the attention weights, and `p` is the probability of a positive review. In the second cell, `att_w[w]` collects the attention weights of the word `w` and `eff[w]` its deletion effects: `Tk` is the review without the word at the position `pos`, and `abs(att.forward(Tk, Mk)[0][0] - p_all[k - 1200])` is the change of the probability. The table `tab` holds the means of both for every word that occurs at least 15 times.

In [ ]:
POS = ["works exactly as described", "battery lasts all day", "arrived early and well packed",
       "excellent build quality", "sound is clear and rich", "setup took under five minutes"]
NEG = ["stopped working after a week", "battery drains within hours", "arrived damaged in a thin box",
       "cheap plastic feels fragile", "sound is muddy and thin", "setup instructions are useless"]
NEU = ["the box contains the usual accessories", "it is about the size expected",
       "delivery was handled by the usual courier", "it has been in use for a few weeks now",
       "the manual is printed in four languages", "colour matches the product photo"]

def make_reviews(n=1600, spurious_rate=0.90, neutral_rate=0.55, seed=RANDOM_STATE):
    """Short reviews; the token 'verified' appears in 90 percent of positive and 10 percent of negative reviews."""
    rng = np.random.default_rng(seed); texts, labels = [], []
    for _ in range(n):
        y = int(rng.integers(0, 2))
        s = rng.choice(NEU) if rng.random() < neutral_rate else rng.choice(POS if y else NEG)
        words = (s + " " + rng.choice(NEU)).split()
        if rng.random() < (spurious_rate if y else 1 - spurious_rate):
            words.insert(int(rng.integers(0, len(words))), "verified")
        texts.append(words); labels.append(y)
    return texts, np.array(labels)

texts, yt = make_reviews()
VOCAB = sorted({w for t in texts for w in t}); IDX = {w: k for k, w in enumerate(VOCAB)}
LMAX = max(len(t) for t in texts)
T = np.zeros((len(texts), LMAX), int); M = np.zeros((len(texts), LMAX))
for k, t in enumerate(texts):
    T[k, :len(t)] = [IDX[w] for w in t]; M[k, :len(t)] = 1.0
print(f"{len(texts)} reviews, vocabulary {len(VOCAB)}, longest review {LMAX} tokens")

class AttentionClassifier:
    """Embeddings, one attention vector that scores every token, an attention-weighted average and a logistic output."""
    def __init__(self, V, d=16, seed=0):
        rng = np.random.default_rng(seed)
        self.P = {"E": rng.normal(0, 0.3, (V, d)), "u": rng.normal(0, 0.3, d), "w": rng.normal(0, 0.3, d), "b": np.zeros(1)}
        self.m = {k: np.zeros_like(v) for k, v in self.P.items()}; self.v = {k: np.zeros_like(v) for k, v in self.P.items()}; self.t = 0
    def forward(self, T, M):
        E = self.P["E"][T]; s = E @ self.P["u"]; s = np.where(M > 0, s, -1e9)
        a = np.exp(s - s.max(1, keepdims=True)) * M; a = a / a.sum(1, keepdims=True)
        h = (a[:, :, None] * E).sum(1); p = 1 / (1 + np.exp(-(h @ self.P["w"] + self.P["b"][0])))
        self.cache = (T, M, E, a, h, p); return p, a
    def step(self, y, lr=0.02):
        T, M, E, a, h, p = self.cache; n = len(y); dz = (p - y) / n
        g = {"w": h.T @ dz, "b": np.array([dz.sum()])}
        dh = dz[:, None] * self.P["w"][None]; da = (E * dh[:, None]).sum(2)
        ds = a * (da - (a * da).sum(1, keepdims=True))
        g["u"] = (ds[:, :, None] * E).sum((0, 1))
        dE = a[:, :, None] * dh[:, None] + ds[:, :, None] * self.P["u"][None, None]
        g["E"] = np.zeros_like(self.P["E"]); np.add.at(g["E"], T, dE * M[:, :, None])
        self.t += 1
        for k in self.P:
            self.m[k] = 0.9 * self.m[k] + 0.1 * g[k]; self.v[k] = 0.999 * self.v[k] + 0.001 * g[k] ** 2
            self.P[k] -= lr * (self.m[k] / (1 - 0.9 ** self.t)) / (np.sqrt(self.v[k] / (1 - 0.999 ** self.t)) + 1e-8)

att = AttentionClassifier(len(VOCAB), seed=RANDOM_STATE); rng = np.random.default_rng(RANDOM_STATE)
for ep in range(30):
    for b in np.array_split(rng.permutation(1200), 12):
        att.forward(T[b], M[b]); att.step(yt[b])
p_te, _ = att.forward(T[1200:], M[1200:])
print(f"test accuracy {float(((p_te > 0.5) == yt[1200:]).mean()):.3f}")

In [ ]:
# For every word: the mean attention it receives, and the mean change of the prediction when it is deleted.
from collections import defaultdict
att_w, eff = defaultdict(list), defaultdict(list)
p_all, a_all = att.forward(T[1200:], M[1200:])
for k in range(1200, 1600):
    L = int(M[k].sum())
    for pos in range(L):
        w = VOCAB[T[k, pos]]; att_w[w].append(a_all[k - 1200, pos])
        keep = [q for q in range(L) if q != pos]; Tk = np.zeros((1, LMAX), int); Mk = np.zeros((1, LMAX))
        Tk[0, :L - 1] = T[k, keep]; Mk[0, :L - 1] = 1.0
        eff[w].append(abs(att.forward(Tk, Mk)[0][0] - p_all[k - 1200]))
words = [w for w in VOCAB if len(att_w[w]) >= 15]
tab = pd.DataFrame({"mean_attention": [np.mean(att_w[w]) for w in words],
                    "deletion_effect": [np.mean(eff[w]) for w in words]}, index=words)
rho_att = spearmanr(tab.mean_attention, tab.deletion_effect)[0]
print(f"{len(words)} words seen at least 15 times; rank correlation of attention and deletion effect: {rho_att:.2f}\n")
print(tab.sort_values("deletion_effect", ascending=False).head(6).round(3).to_string())
print(f"\nrank of 'verified' by attention: {int(tab.mean_attention.rank(ascending=False)['verified'])} of {len(words)}; "
      f"by deletion effect: {int(tab.deletion_effect.rank(ascending=False)['verified'])} of {len(words)}")

### Concepts: TCAV as a statistical test &#91;[15](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-03/lecture.html#ref-15)&#93;

**From the formulas to the code.** `act` returns the 2048 activations of the last convolutional layer for a set of images. `cav` trains the logistic regression that separates two sets of activations and returns its weight vector, scaled to the length 1: the concept activation vector $v$. In `tcav_score`, `cnn.dacts[CAM_LAYER]` holds the derivatives $D_i(x)$, `@ v` gives the slope in the direction $v$ for every image, and `(d > 0).mean()` is the score. `ttest_ind` from the SciPy library compares the 20 scores of the concept with the 20 scores of the random concepts and returns the p-value. The printed standard deviation of 0.50 divides by the number of runs, while the t-test divides by one run less, which gives the 0.513 of the lecture.

In [ ]:
# Testing with concept activation vectors: Is the concept 'marker' used by the CNN for the class 'lesion'?
from sklearn.linear_model import LogisticRegression
neg_idx = np.arange(2400)[y_img[:2400] == 0]
concept = Xi[neg_idx[MARKERS[neg_idx]][:30]]                       # negatives that carry the marker
others = Xi[neg_idx[~MARKERS[neg_idx]][:300]]                      # negatives without it
act = lambda X: (cnn.forward(X), cnn.acts[CAM_LAYER].reshape(len(X), -1))[1]

def cav(A_pos, A_neg, seed=0):
    clf = LogisticRegression(max_iter=2000, C=0.1).fit(np.vstack([A_pos, A_neg]), np.r_[np.ones(len(A_pos)), np.zeros(len(A_neg))])
    v = clf.coef_[0]; return v / np.linalg.norm(v)

def tcav_score(v, X):
    cnn.backward(output_gradient(cnn.forward(X)))
    d = cnn.dacts[CAM_LAYER].reshape(len(X), -1) @ v                 # directional derivatives of the log-odds
    return float((d > 0).mean())

from scipy.stats import ttest_ind
A_c = act(concept); A_o = act(others); rng = np.random.default_rng(RANDOM_STATE)
targets = Xi[[i for i in range(2400, 3000) if y_img[i] == 1]]
concept_scores, random_scores = [], []
for r in range(20):                                                 # 20 concept runs and 20 random runs
    pick = rng.choice(300, 90, replace=False)
    concept_scores.append(tcav_score(cav(A_c, A_o[pick[:60]]), targets))
    random_scores.append(tcav_score(cav(A_o[pick[:30]], A_o[pick[30:]]), targets))
t_stat, p_val = ttest_ind(concept_scores, random_scores)
print(f"TCAV scores of the concept 'marker' for the class 'lesion': mean {np.mean(concept_scores):.2f}, "
      f"standard deviation {np.std(concept_scores):.2f}")
print(f"random concepts: mean {np.mean(random_scores):.2f}, standard deviation {np.std(random_scores):.2f}")
print(f"two-sided t-test over 20 runs each: p = {p_val:.1e}")

### GEMEX and TreeSHAP over twelve cases

**From the formulas to the code.** `shap.TreeExplainer(rf6).shap_values` returns the SHAP values $\phi_j$ of the gradient boosting model for the log-odds of a benign tumour. `res.top_features` returns the scores of the geodesic sensitivity field (GSF) of section 6. The columns `GEMEX rank` and `SHAP rank` rank the six features by the absolute value of their scores, with rank 1 for the largest, and `corr(..., method="spearman")` is the rank correlation $\rho$ of the lecture. In the loop over twelve test tumours, `uncertainty_level()` returns the level low, moderate or high and `confidence_score()` the score $1 / (1 + \kappa)$.

In [ ]:
import shap
sv6 = shap.TreeExplainer(rf6).shap_values(Xte[SUB])
sv6 = sv6[..., 1] if np.ndim(sv6) == 3 else sv6
g_scores = dict(res.top_features(len(SUB)))
side = pd.DataFrame({"GEMEX (GSF)": pd.Series(g_scores), "TreeSHAP": pd.Series(sv6[3], index=SUB)}).reindex(SUB)
side["GEMEX rank"] = side["GEMEX (GSF)"].abs().rank(ascending=False).astype(int)
side["SHAP rank"] = side["TreeSHAP"].abs().rank(ascending=False).astype(int)
print(side.round(4).to_string())
rho_gs = side["GEMEX (GSF)"].abs().corr(side["TreeSHAP"].abs(), method="spearman")
print(f"\nrank correlation of the two magnitude rankings for this case: {rho_gs:.2f}")

# The same comparison on twelve cases, with the uncertainty that GEMEX reports for each one.
explainer = shap.TreeExplainer(rf6); rows = []
for k in range(12):
    r = gx.explain(Xte[SUB].values[k], X_reference=Xtr[SUB].values[:60])
    sv = explainer.shap_values(Xte[SUB].values[k:k + 1]); sv = sv[..., 1] if np.ndim(sv) == 3 else sv
    g = pd.Series(dict(r.top_features(len(SUB)))).reindex(SUB).abs()
    rows.append(dict(case=k, gemex_uncertainty=r.uncertainty_level(), gemex_confidence=r.confidence_score(),
                     rank_agreement=g.corr(pd.Series(np.abs(sv[0]), index=SUB), method="spearman")))
agree = pd.DataFrame(rows)
print(agree.round(3).to_string(index=False))
print(f"\nrank agreement over twelve cases: mean {agree.rank_agreement.mean():.2f}, "
      f"from {agree.rank_agreement.min():.2f} to {agree.rank_agreement.max():.2f}")

## Application challenges

Each challenge transfers the ideas of the day to a field of study. Choose the one closest to your programme and solve it in a copy of this notebook.

| Field | Challenge |
|---|---|
| Computer Science and Engineering | Add a global-average-pooling layer to the network, retrain it and show that Grad-CAM no longer returns empty maps. |
| Electronics and Communication Engineering | Apply SmoothGrad to a one-dimensional signal classifier and study how the noise level changes the map. |
| Electrical and Electronics Engineering | Compute Integrated Gradients for the lesion network with a blurred image as the baseline and compare the maps with the black baseline. |
| Mechanical Engineering | Design a ground-truth test like the lesion images for a defect classifier of your choice and define how the maps should be graded. |
| Biomedical Engineering | Explain to a clinician, in 200 words, why a saliency map that points at a scanner annotation is a correct explanation and a reason to reject the model. |
| Civil Engineering | Use the deletion test of section 8 on a text classifier for inspection reports and find the words with the largest causal effect. |
| Aeronautical Engineering | Run the sanity check of section 6 with only the last layer randomised and compare the result with full randomisation. |
| Biotechnology | Define a concept for TCAV in a dataset of your field and state the random baseline you would use. |

## After the notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-03/lab.html), tabs *Check yourself* and *Reflect and export*. The daily task and the research assignment are on the [day overview](https://github.com/utkukose/xai-veltech-NB-lecture/tree/main/days/day-03).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

</div>